In [2]:
# Cell 1 — Check PyTorch and GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (compiled): {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected — PyTorch will fall back to CPU (much slower)")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU detected: NVIDIA GeForce RTX 3050 Ti Laptop GPU
CUDA version (compiled): 12.1
GPU memory: 4.29 GB


In [4]:
# Cell 2 — Load a pre-trained MobileNetV3 and run it on a sample image
import torch
import timm
from PIL import Image
import urllib.request
import io

# 1. Pick the GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load MobileNetV3-Small, pre-trained on ImageNet (1000 categories)
print("\nLoading MobileNetV3-Small (pre-trained on ImageNet)...")
model = timm.create_model("mobilenetv3_small_100", pretrained=True)
model = model.to(device)
model.eval()  # inference mode (turns off training-specific layers)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# 3. Download a sample image (a cat) and prepare it for the model
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
print(f"\nDownloading sample image from {url}...")
with urllib.request.urlopen(url) as response:
    img = Image.open(io.BytesIO(response.read())).convert("RGB")
print(f"Image size: {img.size}")

# 4. Get the proper preprocessing pipeline for this model
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

# 5. Apply preprocessing → tensor, add batch dimension, send to GPU
input_tensor = transform(img).unsqueeze(0).to(device)
print(f"Input tensor shape: {input_tensor.shape}")

# 6. Run inference (no_grad = don't track gradients, saves memory)
with torch.no_grad():
    output = model(input_tensor)

# 7. Convert to probabilities and get the top-5 predictions
probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_indices = torch.topk(probabilities, 5)

# 8. Load ImageNet class labels so we can show readable names
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
with urllib.request.urlopen(labels_url) as response:
    categories = [line.strip().decode("utf-8") for line in response.readlines()]

print("\nTop 5 predictions:")
for prob, idx in zip(top5_prob, top5_indices):
    print(f"  {categories[idx]:30s} {prob.item()*100:.2f}%")

Using device: cuda

Loading MobileNetV3-Small (pre-trained on ImageNet)...


model.safetensors:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

C:\Users\Shantanu\miniconda3\envs\deepfake\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shantanu\.cache\huggingface\hub\models--timm--mobilenetv3_small_100.lamb_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model loaded. Parameters: 2,542,856

Image size: (1546, 1213)
Input tensor shape: torch.Size([1, 3, 224, 224])

Top 5 predictions:
  Samoyed                        60.97%
  Pomeranian                     5.05%
  Arctic fox                     3.31%
  wallaby                        1.79%
  papillon                       1.21%
